# Part 6: Live Demo — dataset upload করে botnet detect করা
**Robust IoT Botnet Detection Under Incomplete Network Traffic and Cross-Device Distribution Shifts**

এই notebook-এ BRB model একবার train হয় (প্রায় ১-২ মিনিট)। তারপর:
1. **Demo A:** তৈরি করা নমুনা (সুস্থ + আক্রান্ত traffic) দিয়ে detect করে দেখায়, চাইলে কিছু feature মুছে দিয়েও।
2. **Demo B:** তুমি নিজের CSV upload করো, প্রতিটা সারির জন্য botnet-এর সম্ভাবনা ও কারণ (rule) দেখো।

**Run:** Runtime → Run all। সবচেয়ে উপরের সেটিং cell-এ শুধু ৩টা জিনিস বদলানো যায়।

## 1. সেটিং (এখানে শুধু এই ক'টা বদলাও)

In [ ]:
SOURCE = 1                 # যে device-এর traffic দিয়ে model শেখানো হবে (1,2,4,5,6,8,9)
DEMO_DEVICE = 2            # Demo A-তে কোন device-এর traffic পরীক্ষা হবে (SOURCE থেকে আলাদা দাও)
DEMO_ATTACK = "gafgyt"     # "gafgyt" (নতুন ভাইরাস) বা "mirai" (চেনা ভাইরাস)
DEMO_MISSING = 0.3         # Demo A-তে কত শতাংশ feature মুছে দেওয়া হবে (0.0 = কিছু না, 0.5 = অর্ধেক)
THRESHOLD = 0.5            # P(botnet) এর উপরে হলে "BOTNET" বলা হবে

## 2. Setup ও dataset

In [ ]:
import glob, os, itertools, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
SEED = 42
N_PER_CLASS = 20000
SELECTED = ['H_L0.01_weight', 'H_L3_weight', 'HpHp_L0.1_weight',
            'HH_jit_L0.01_mean', 'MI_dir_L5_variance', 'HpHp_L0.01_radius']

In [ ]:
!pip install -q kagglehub
import kagglehub
path = kagglehub.dataset_download("mkashifn/nbaiot-dataset")
print(path)

In [ ]:
def find_files(dev, kind):
    return sorted(glob.glob(f"{path}/**/{dev}.{kind}*.csv", recursive=True))

def read(f):
    return pd.read_csv(f, usecols=SELECTED)[SELECTED].astype("float32")

def load_benign(dev, n=N_PER_CLASS, seed=SEED):
    d = pd.concat([read(f) for f in find_files(dev, "benign")], ignore_index=True)
    return d.sample(min(len(d), n), random_state=seed)

def load_attack(dev, family, n=N_PER_CLASS, seed=SEED):
    files = find_files(dev, family)
    assert files, f"device {dev}-এ {family} নেই"
    per = max(1, n // len(files))
    parts = []
    for f in files:
        d = read(f)
        parts.append(d.sample(min(len(d), per), random_state=seed))
    return pd.concat(parts, ignore_index=True)

def make_xy(ben, att):
    X = np.vstack([ben.values, att.values]).astype("float64")
    y = np.r_[np.zeros(len(ben)), np.ones(len(att))]
    return X, y

def build_scenario(family, dev):
    """Cross-device scenario: target device-এর benign + নতুন device-এর attack।"""
    ben = load_benign(dev, 10000).values.astype("float64")
    att = load_attack(dev, "mirai" if family == "Mirai" else "gafgyt", 10000).values.astype("float64")
    return np.vstack([ben, att]), np.r_[np.zeros(len(ben)), np.ones(len(att))]

In [ ]:
GRADES = ["Low", "Medium", "High"]

def slog(X):
    return np.sign(X) * np.log1p(np.abs(X))

class BRB:
    def __init__(self, d, lam=5.0):
        self.d, self.lam = d, lam
        self.idx = np.array(list(itertools.product(range(3), repeat=d)))
        self.K = len(self.idx)

    def match(self, Z, refs):
        n, d = Z.shape
        M = np.empty((n, d, 3))
        for i in range(d):
            r0, r1, r2 = refs[i]
            z = np.clip(Z[:, i], r0, r2)
            t1 = (z - r0) / (r1 - r0 + 1e-9)
            t2 = (z - r1) / (r2 - r1 + 1e-9)
            lo = z <= r1
            M[:, i, 0] = np.where(lo, 1 - t1, 0)
            M[:, i, 1] = np.where(lo, t1, 1 - t2)
            M[:, i, 2] = np.where(lo, 0, t2)
        return np.clip(M, 0, 1)

    def fill(self, M, Z):
        miss = np.isnan(Z)
        if not miss.any():
            return M
        return np.where(miss[:, :, None], np.ones((1, 1, 3)), M)   # missing = ignorance

    def activation(self, M, delta):
        P = M ** delta[None, :, None]
        A = P[:, 0, :]
        for i in range(1, self.d):
            A = (A[:, :, None] * P[:, i, :][:, None, :]).reshape(len(A), -1)
        return A / (A.sum(1, keepdims=True) + 1e-12)

    def fit_beliefs(self, M, y, delta, chunk=4000):
        S = np.zeros(self.K); C1 = np.zeros(self.K)
        for s in range(0, len(y), chunk):
            A = self.activation(M[s:s + chunk], delta)
            S += A.sum(0); C1 += A.T @ y[s:s + chunk]
        return np.stack([S - C1, C1], 1) / (S[:, None] + self.lam)

    def infer(self, M, delta, beta, chunk=3000):
        s = beta.sum(1); out = []
        for st in range(0, len(M), chunk):
            A = self.activation(M[st:st + chunk], delta)
            one_m = 1 - A * s[None, :]
            f = A[:, :, None] * beta[None, :, :] + one_m[:, :, None]
            P = f.prod(1); base = one_m.prod(1); wprod = (1 - A).prod(1)
            mu = 1.0 / (P.sum(1) - base + 1e-12)
            out.append(mu[:, None] * (P - base[:, None]) / (1 - mu * wprod + 1e-12)[:, None])
        b = np.clip(np.vstack(out), 0, 1)
        unk = np.clip(1 - b.sum(1), 0, 1)
        return b[:, 1] + 0.5 * unk, b

brb = BRB(len(SELECTED))
print("Rule সংখ্যা:", brb.K)

## 3. Model train করা (একবারই)

In [ ]:
d = len(SELECTED)
ONES = np.ones(d)
Q = np.array([[0.05, 0.5, 0.95]] * d)
def make_refs(Z): return np.array([np.quantile(Z[:, i], Q[i]) for i in range(d)])

XA, yA = make_xy(load_benign(SOURCE), load_attack(SOURCE, "mirai"))
Ztr = slog(XA)
refs = make_refs(Ztr)
beta = brb.fit_beliefs(brb.match(Ztr, refs), yA, ONES)
print(f"Model তৈরি। Source device {SOURCE}, training নমুনা {len(yA)} (সুস্থ + Mirai)।")

def detect(df, threshold=THRESHOLD):
    """df-এ SELECTED কলামগুলো থাকতে হবে। ফাঁকা/NaN মানে 'missing'।"""
    X = df[SELECTED].astype("float64").values
    Z = slog(X)
    p, b = brb.infer(brb.fill(brb.match(Z, refs), Z), ONES, beta)
    out = df.copy()
    out["P_botnet"] = p
    out["ignorance"] = np.clip(1 - b.sum(1), 0, 1)
    out["missing_features"] = np.isnan(X).sum(1)
    out["verdict"] = np.where(p >= threshold, "BOTNET", "normal")
    return out

def explain(row, top=3):
    """একটা সারির জন্য কোন rule-গুলো সবচেয়ে বেশি কাজ করেছে।"""
    x = np.asarray(row[SELECTED], dtype="float64")[None, :]
    Z = slog(x); miss = np.isnan(Z[0])
    A = brb.activation(brb.fill(brb.match(Z, refs), Z), ONES)[0]
    vis = np.where(~miss)[0]
    groups = {}
    for k in np.where(A > 1e-6)[0]:
        key = tuple(brb.idx[k][vis])
        w, bt = groups.get(key, (0.0, np.zeros(2)))
        groups[key] = (w + A[k], bt + A[k] * beta[k])
    print("Missing feature:", [SELECTED[i] for i in np.where(miss)[0]] or "নেই")
    for key, (w, bt) in sorted(groups.items(), key=lambda kv: -kv[1][0])[:top]:
        bt = bt / w
        cond = " AND ".join(f"{SELECTED[i]} is {GRADES[g]}" for i, g in zip(vis, key))
        print(f"  IF {cond}\n    THEN Normal={bt[0]:.2f}, Botnet={bt[1]:.2f}, Ignorance={max(0, 1-bt.sum()):.2f}   (rule weight {w:.3f})")

## 4. Demo A — তৈরি নমুনা দিয়ে detect
Device `DEMO_DEVICE`-এর ১০০০টা সুস্থ + ১০০০টা আক্রান্ত নমুনা, `DEMO_MISSING` শতাংশ feature মুছে।
Model এই device বা এই ভাইরাস দিয়ে **শেখেনি** (Gafgyt হলে একদম নতুন ভাইরাস)।

In [ ]:
assert DEMO_DEVICE != SOURCE, "DEMO_DEVICE আর SOURCE আলাদা দাও"
X, y = build_scenario("Mirai" if DEMO_ATTACK == "mirai" else "Gafgyt", DEMO_DEVICE)
rng = np.random.default_rng(0)
idx = np.r_[rng.choice(np.where(y == 0)[0], 1000, replace=False), rng.choice(np.where(y == 1)[0], 1000, replace=False)]
rng.shuffle(idx)
demo = pd.DataFrame(X[idx], columns=SELECTED); ytrue = y[idx]
if DEMO_MISSING > 0:
    demo = demo.mask(rng.random(demo.shape) < DEMO_MISSING)
demo.assign(label=ytrue).to_csv("sample_demo.csv", index=False)   # পরে Demo B-তে upload করে দেখা যাবে

res = detect(demo)
pred = (res["P_botnet"].values >= THRESHOLD).astype(int)
tp = int(((pred == 1) & (ytrue == 1)).sum()); fn = int(((pred == 0) & (ytrue == 1)).sum())
fp = int(((pred == 1) & (ytrue == 0)).sum()); tn = int(((pred == 0) & (ytrue == 0)).sum())
print(f"Device {DEMO_DEVICE}, {DEMO_ATTACK}, {int(DEMO_MISSING*100)}% feature missing")
print(f"মোট নমুনা: {len(res)}  (আসলে আক্রান্ত {int(ytrue.sum())}, সুস্থ {int((ytrue==0).sum())})")
print(f"BOTNET বলেছে: {int(pred.sum())}")
print(f"  ঠিক ধরেছে (আক্রান্তকে BOTNET): {tp}   মিস করেছে: {fn}")
print(f"  ভুল সতর্কতা (সুস্থকে BOTNET): {fp}   ঠিক সুস্থ বলেছে: {tn}")
print(f"AUC (threshold-ছাড়া মান): {roc_auc_score(ytrue, res['P_botnet']):.3f}")
plt.figure(figsize=(6, 3))
plt.hist(res["P_botnet"][ytrue == 0], bins=30, alpha=.6, label="আসলে সুস্থ")
plt.hist(res["P_botnet"][ytrue == 1], bins=30, alpha=.6, label="আসলে আক্রান্ত")
plt.axvline(THRESHOLD, color="k", ls="--"); plt.xlabel("P(botnet)"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
print("=== ধরা পড়া একটা botnet ===")
hit = np.where((pred == 1) & (ytrue == 1))[0]
if len(hit): print(f"P(botnet) = {res.iloc[hit[0]]['P_botnet']:.3f}"); explain(demo.iloc[hit[0]])
print("\n=== মিস হওয়া একটা botnet ===")
miss_i = np.where((pred == 0) & (ytrue == 1))[0]
if len(miss_i): print(f"P(botnet) = {res.iloc[miss_i[0]]['P_botnet']:.3f}"); explain(demo.iloc[miss_i[0]])

## 5. Demo B — নিজের CSV upload করো
**নিয়ম:** CSV-তে অন্তত এই ৬টা কলাম থাকতে হবে (N-BaIoT-এর আসল কলামের নাম): `H_L0.01_weight, H_L3_weight, HpHp_L0.1_weight, HH_jit_L0.01_mean, MI_dir_L5_variance, HpHp_L0.01_radius`।
- N-BaIoT-এর আসল ফাইল (যেমন `2.gafgyt.combo.csv`) দিলে হবে। ফাইল বড় হলে প্রথম ৫০০০ সারি নেওয়া হয়।
- ফাইলের নামে `benign`, `mirai` বা `gafgyt` থাকলে আমি আসল label ধরে নিয়ে নির্ভুলতা দেখাব। না থাকলে শুধু ভবিষ্যদ্বাণী দেখাবে।
- কিছু upload না করলে Demo A-র `sample_demo.csv` ব্যবহার হবে।
- খালি ঘর বা NaN = missing feature।

In [ ]:
try:
    from google.colab import files
    up = files.upload()
    fname = next(iter(up)) if up else None
except Exception:
    fname = None
if not fname:
    fname = "sample_demo.csv"; print("কিছু upload হয়নি, sample_demo.csv ব্যবহার করছি")
head = pd.read_csv(fname, nrows=0)
absent = [c for c in SELECTED if c not in head.columns]
assert not absent, f"এই কলামগুলো CSV-তে নেই: {absent}"
usecols = SELECTED + (["label"] if "label" in head.columns else [])
udf = pd.read_csv(fname, usecols=usecols)
if len(udf) > 5000: udf = udf.sample(5000, random_state=SEED).reset_index(drop=True)
ulabel = None
if "label" in udf.columns: ulabel = udf["label"].values.astype(int)
else:
    nm = os.path.basename(fname).lower()
    if "benign" in nm: ulabel = np.zeros(len(udf), int)
    elif "mirai" in nm or "gafgyt" in nm: ulabel = np.ones(len(udf), int)
r = detect(udf)
n_bot = int((r["verdict"] == "BOTNET").sum())
print(f"ফাইল: {fname}, পরীক্ষিত সারি: {len(r)}")
print(f"BOTNET: {n_bot} ({100*n_bot/len(r):.1f}%),  সুস্থ: {len(r)-n_bot}")
if ulabel is not None:
    ok = ((r["verdict"] == "BOTNET").astype(int).values == ulabel).mean()
    print(f"আসল label-এর সঙ্গে মিল: {100*ok:.1f}%" + (f",  AUC: {roc_auc_score(ulabel, r['P_botnet']):.3f}" if len(set(ulabel)) == 2 else ""))
display(r.sort_values("P_botnet", ascending=False).head(10)[SELECTED[:2] + ["P_botnet", "ignorance", "missing_features", "verdict"]].round(3))
plt.figure(figsize=(6, 3)); plt.hist(r["P_botnet"], bins=30); plt.axvline(THRESHOLD, color="k", ls="--")
plt.xlabel("P(botnet)"); plt.ylabel("সারির সংখ্যা"); plt.tight_layout(); plt.show()

In [ ]:
top = r["P_botnet"].idxmax()
print(f"সবচেয়ে সন্দেহজনক সারি #{top}: P(botnet) = {r.loc[top, 'P_botnet']:.3f}")
explain(udf.loc[top])

---
## মনে রাখার কথা
- নতুন ভাইরাসে (Gafgyt) `0.5` threshold ঠিকমতো মেলে না, কিছু আক্রান্ত মিস হতে পারে। তাই **AUC** আসল মাপ।
- নমুনা আসল traffic নয়, N-BaIoT dataset-এর রেকর্ড। এটা গবেষণার demo, বাস্তব network-এ সরাসরি চালানোর সিস্টেম নয়।